Attention divides its scores by $\sqrt{d_k}$, where $d_k$ is the dimension of the query and key vectors of one head. In standard equal-width multi-head attention $d_k = d/H$, not the model width $d$.

The reason is the variance of a dot product. Assume the components of a query $q$ and a key $k$ are independent, with mean 0 and variance 1, an idealized initialization model, not a guarantee for learned projections. Each product $q_\ell k_\ell$ then has mean 0 and variance 1, and the score is a sum of $d_k$ such terms:

$$
s = q \cdot k = \sum_{\ell=1}^{d_k} q_\ell k_\ell, \qquad \mathbb{E}[s] = 0, \qquad \mathrm{Var}(s) = d_k .
$$

The scores spread out in proportion to $\sqrt{d_k}$. Dividing by $\sqrt{d_k}$ brings the variance back to 1 for any head size:

$$
\mathrm{Var}\!\left(\frac{q \cdot k}{\sqrt{d_k}}\right) = 1 .
$$

More generally, if the independent zero-mean components have variances $\sigma_q^2$ and $\sigma_k^2$, then $\operatorname{Var}(q_\ell k_\ell)=\mathbb E[q_\ell^2]\mathbb E[k_\ell^2]=\sigma_q^2\sigma_k^2$. Assuming independence across components, $\operatorname{Var}(q\cdot k)=d_k\sigma_q^2\sigma_k^2$. For equal variance $\sigma^2$, this is $d_k\sigma^4$. Scaling removes the factor $d_k$, not the dependence on component variance. Correlated components or queries and keys need not obey this formula exactly.

The next cell checks both variances by sampling 10,000 query and key pairs for each head size.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
dims = [8, 16, 32, 64, 128, 256, 512]
var_raw, var_scaled = [], []
for d_k in dims:
    q = rng.standard_normal((10_000, d_k))
    k = rng.standard_normal((10_000, d_k))
    s = (q * k).sum(axis=1)
    var_raw.append(s.var())
    var_scaled.append((s / np.sqrt(d_k)).var())
    print(f"d_k = {d_k:3d}   Var(q.k) = {var_raw[-1]:7.1f}   Var(q.k / sqrt(d_k)) = {var_scaled[-1]:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(dims, var_raw, marker="o", label=r"$q \cdot k$")
ax.plot(dims, var_scaled, marker="o", label=r"$q \cdot k / \sqrt{d_k}$")
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xlabel("$d_k$")
ax.set_ylabel("variance of the score")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Large score differences can make softmax concentrate on one key. Its Jacobian is $J=\operatorname{diag}(a)-aa^\top$, where $a$ is the attention-weight vector. When $a$ approaches a one-hot vector, this Jacobian approaches zero. This can weaken gradients through the query-key matching path, but does not imply that all model gradients vanish. The experiment below uses one query and 16 keys with $d_k = 64$. It multiplies the scaled scores by a factor $c$ and tracks the largest attention weight and the norm of the Jacobian. Unscaled scores are $\sqrt{d_k}$ times larger than scaled ones, so $c = 1$ is scaled attention and $c = \sqrt{d_k} = 8$ is unscaled attention.

In [ ]:
def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()


def jacobian_norm(w):
    return np.linalg.norm(np.diag(w) - np.outer(w, w))


d_k, T = 64, 16
q = rng.standard_normal(d_k)
K = rng.standard_normal((T, d_k))
z = K @ q / np.sqrt(d_k)  # scaled scores, variance about 1

factors = np.logspace(-1, 2, 61)
largest = [softmax(c * z).max() for c in factors]
norms = [jacobian_norm(softmax(c * z)) for c in factors]

for c in (1, 3, 8, 16, 32):
    w = softmax(c * z)
    print(f"c = {c:2d}: largest weight {w.max():.3f}, keys above 0.01: {(w > 0.01).sum():2d} of {T}, "
          f"Jacobian norm {jacobian_norm(w):.3f}")

w_scaled = softmax(z)
w_raw = softmax(np.sqrt(d_k) * z)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, w, title in zip(axes, [w_raw, w_scaled], [r"$\mathrm{softmax}(q \cdot k)$", r"$\mathrm{softmax}(q \cdot k / \sqrt{d_k})$"]):
    ax.bar(range(T), w)
    ax.set_title(title)
    ax.set_xlabel("key index")
axes[0].set_ylabel("attention weight")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 3.5))
ax1.plot(factors, norms, color="tab:blue", label="Jacobian norm")
ax1.set_xscale("log")
ax1.set_xlabel("score multiplier $c$")
ax1.set_ylabel("Jacobian norm", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(factors, largest, color="tab:orange", label="largest weight")
ax2.set_ylabel("largest weight", color="tab:orange")
ax2.set_ylim(0, 1.05)
for c, label in [(1, "scaled"), (np.sqrt(d_k), "unscaled")]:
    ax1.axvline(c, color="gray", ls="--", lw=1)
    ax1.text(c * 1.08, max(norms) * 0.95, label, color="gray", fontsize=9)
ax1.grid(alpha=0.3)
plt.tight_layout()
plt.show()

The Jacobian norm peaks at a moderate spread of the weights, around $c = 3$, and is small at both ends, for different reasons. When the weights are nearly equal (small $c$), each entry of $J$ is of order $1/T$, so the norm stays near 0.24 but does not vanish. Past the peak the norm falls fast as one weight approaches 1: about 0.25 at $c = 8$, 0.05 at $c = 16$, and close to zero beyond $c = 32$.

Unscaled scores correspond to $c = \sqrt{d_k}$, so these three points are unscaled heads with $d_k = 64$, 256 and 1024. The larger the head, the further unscaled attention sits in the region where the gradient through the softmax vanishes. Scaling keeps $c = 1$ for every head size.

## PyTorch reference

| PyTorch class | Description |
|---|---|
| [`nn.Linear`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.Linear.html) | Applies an affine linear transformation to the incoming data: $y = xA^T + b$. |
| [`nn.Softmax`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.Softmax.html) | Applies the Softmax function to an n-dimensional input Tensor. |
| [`nn.MultiheadAttention`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.MultiheadAttention.html) | Allows the model to jointly attend to information from different representation subspaces. |

Reference: Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762), Section 3.2.1.